In [1]:
import os
import time
import numpy as np
import tensorflow as tf
import tf2onnx   # Convert TensorFlow/Keras to ONNX
import onnx      # Inspect and validate the ONNX model
import onnxruntime as ort   # Execute ONNX models

print("TensorFlow:", tf.__version__)
print("ONNX:", onnx.__version__)
print("ONNX Runtime:", ort.__version__)
print("tf2onnx:", tf2onnx.__version__)

2026-11-05 20:29:59.002638: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow: 2.20.0
ONNX: 1.23.2
ONNX Runtime: 1.23.2
tf2onnx: 1.17.0


## Load tensorflow model

In [2]:
tf_model = tf.keras.models.load_model(
    "best_resnet50_finetuned.keras",
    compile=False
)

tf_model.summary()

Model: "Cattle_Buffalo_Breed_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_avg_pooling              │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_1                    │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_2                    │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ breed_classification (Dense)    │ (None, 14)             │         3,598 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,781,966 (94.54 MB)

 Trainable params: 11,179,278 (42.65 MB)

 Non-trainable params: 13,602,688 (51.89 MB)

In [3]:
print("Input shape:", tf_model.input_shape)
print("Output shape:", tf_model.output_shape)

Input shape: (None, 224, 224, 3)
Output shape: (None, 14)


## Convert into ONNX format

In [4]:
input_signature = [
    tf.TensorSpec(
        shape=(None, *tf_model.input_shape[1:]),
        dtype=tf.float32,
        name = "input"
    )
] # Describes the type of a tf.Tensor.

In [ ]:
onnx_model, _ = tf2onnx.convert.from_keras(
    tf_model,
    input_signature=input_signature,
    opset = 15,
    output_path="best_resnet50_finetuned.onnx"
)
print("Convertion completed")

I0000 00:00:1793881456.746409   30458 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1793881456.746921   30458 single_machine.cc:376] Starting new session
I0000 00:00:1793881468.397627   30458 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1793881468.507945   30458 single_machine.cc:376] Starting new session


## Compare size

In [5]:
tf_file_size = os.path.getsize("./best_resnet50_finetuned.keras") # In Bytes
onnx_file_size = os.path.getsize("./best_resnet50_finetuned.onnx")

tf_file_size = tf_file_size/ (1024*1024) # bytes -> Kilobytes -> Megabytes, that's why (1024*1024)
onnx_file_size = onnx_file_size/ (1024*1024)

print(f"TensorFlow model Size: {tf_file_size:.2f} MB")
print(f"ONNX model Size: {onnx_file_size:.2f} MB")

TensorFlow model Size: 265.83 MB
ONNX model Size: 47.35 MB


## Validate ONNX model

In [6]:
onnx_model = onnx.load("best_resnet50_finetuned.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")

ONNX model is valid.


## Run inference using ONNX Runtime

In [7]:
# Load ONNX model
session = ort.InferenceSession(
    "best_resnet50_finetuned.onnx",
    providers=["CPUExecutionProvider"]
)

In [8]:
# Get input and output information
input_info = session.get_inputs()[0]
output_info = session.get_outputs()[0]

print("Input shape:", input_info.shape)
print("Output shape:", output_info.shape)

Input shape: ['unk__843', 224, 224, 3]
Output shape: ['unk__844', 14]


In [9]:
# Create dummy input
input_data = np.random.rand(
    1, *input_info.shape[1:]
).astype(np.float32)

print(input_data.shape)

(1, 224, 224, 3)


In [10]:
# Perform interface
start = time.perf_counter()

onnx_predictions = session.run(
    None,
    {input_info.name:input_data}
)
end = time.perf_counter()
onnx_time = end-start

In [11]:
# Tensorflow predictions 
start = time.perf_counter()

tf_predictions = tf_model.predict(input_data)

end = time.perf_counter()
tf_time = end-start

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step


In [12]:
print("ONNX Predictions:", onnx_predictions[0])
print("ONNX Predicted class:", np.argmax(onnx_predictions[0]))
print("TF Predictions:", tf_predictions[0])
print("TF Predicted class:", np.argmax(tf_predictions[0]))

ONNX Predictions: [[1.2549866e-02 1.1173734e-02 5.7167765e-03 1.9803736e-02 2.0406181e-03
  1.1453109e-02 6.2050819e-03 1.3124751e-02 1.3124332e-03 8.9473075e-01
  6.6401898e-03 8.7825460e-03 5.7196449e-03 7.4674713e-04]]
ONNX Predicted class: 9
TF Predictions: [1.2587154e-02 1.1090564e-02 5.6870366e-03 1.9924562e-02 2.0545260e-03
 1.1563981e-02 6.1482498e-03 1.3204327e-02 1.3049577e-03 8.9458686e-01
 6.6507920e-03 8.7156789e-03 5.7364884e-03 7.4484653e-04]
TF Predicted class: 9


In [14]:
print("Latency:-")
print(f"TensorFlow runtime: {tf_time} ")
print(f"ONNX runtime: {onnx_time} ")

Latency:-
TensorFlow runtime: 5.095533333005733 
ONNX runtime: 0.451337908001733 


In [6]:
print(f"ONNX model is {tf_time/onnx_time:.2f} times faster than the tensorflow model in CPU.")

ONNX model is 11.29 times faster than the tensorflow model.
